In [1]:
def random_word(difficulty):

    """
    Questa funzione restituisce una parola casuale scelta tra uno di due file di testo depositati sul repository Github pubblico 'paroleitaliane'
    di napolux (Francesco Napoletano) a seconda della difficoltà di gioco scelta dall'utente.
    Modalità facile   : file di 1160 parole italiane più frequenti.
    Modalità difficile: file di 661569 parole italiane, tra cui coniugazioni dei verbi e declinazioni per genere e numero di sostantivi ed aggettivi.
    ---------
    Parameters:
    difficulty, str
        Livello di difficoltà
    ---------
    Returns:
    str
        Parola casuale scelta a seconda del livello di difficoltà impostato
    ---------
    Richiede verifica di input ('difficulty' deve essere "1" per la modalità facile o altro per la modalità difficile).
    """
    
    import requests
    from random import randint
    
    try:
        if difficulty=="facile":
            res = requests.get("https://raw.githubusercontent.com/napolux/paroleitaliane/refs/heads/main/paroleitaliane/1000_parole_italiane_comuni.txt")
        else:
            res = requests.get("https://raw.githubusercontent.com/napolux/paroleitaliane/refs/heads/main/paroleitaliane/660000_parole_italiane.txt")
        res.raise_for_status()
        lst_words = (res.text).splitlines()
        return lst_words[randint(0,len(lst_words)-1)]
        
    except requests.exceptions.HTTPError:
        print("Recupero del dizionario per la generazione casuale della parola da indovinare non riuscito")


In [2]:
def ask_to_play():
    """Se l'utente vuole cominciare a giocare: ``True`` per sì, ``False`` per no.

    La domanda si ripete finché la risposta non è una di quelle riconosciute.
    Una risposta sbagliata non costa una vita: la partita non è ancora
    cominciata, e non ci sono vite da togliere.

    **Risposte riconosciute.** ``s``, ``si`` e ``sì`` per il sì, ``n`` e ``no``
    per il no. Maiuscole e spazi ai lati non contano: ``" Sì "`` vale ``"sì"``.

    **Chi la usa.** Il codice principale, una volta sola, all'avvio. Con
    ``True`` si passa alla scelta della difficoltà (``choose_difficulty``); con
    ``False`` il programma termina senza giocare.
    """
    while True:
        risposta = input("Vuoi giocare all'impiccato? (s/n): ").strip().lower()

        if risposta in ["s", "si", "sì"]:
            return True
        if risposta in ["n", "no"]:
            return False

        # Qui si arriva solo con una risposta non riconosciuta. Il ciclo
        # ripete la domanda: l'unica uscita è uno dei due return qui sopra.
        print("Input non valido: rispondi s (sì) oppure n (no).")

In [3]:
def choose_difficulty():
    """La difficoltà scelta dall'utente, nella forma che ``random_word`` si
    aspetta: ``"facile"`` o ``"difficile"``.

    L'utente risponde con un numero, ``1`` o ``2``; la funzione restituisce la
    parola. Così il resto del codice non deve sapere quale numero corrisponde a
    quale livello: lo sa solo questa funzione.

    **Perché il confronto è fra stringhe.** ``input`` restituisce sempre una
    stringa. Convertirla con ``int`` fermerebbe il programma con un
    ``ValueError`` alla prima lettera digitata; confrontarla con ``"1"`` e
    ``"2"`` tratta una lettera come qualunque altra risposta sbagliata.

    **Chi la usa.** Il codice principale, all'inizio di ogni partita e prima
    di ``random_word``: ``secret_word = random_word(choose_difficulty())``.
    """
    while True:
        # Solo strip, senza lower: le risposte valide sono cifre, e una cifra
        # non ha maiuscole.
        scelta = input("Scegli la difficoltà (1 = facile, 2 = difficile): ").strip()

        if scelta == "1":
            return "facile"
        if scelta == "2":
            return "difficile"

        # Una risposta diversa da 1 e 2 non costa una vita: la partita non è
        # ancora cominciata. Si ripete la domanda.
        print("Input non valido: inserisci 1 oppure 2.")

In [4]:
def handle_error(lives, flag):
    print("Errore!")
    lives -= 1
    
    if lives <= 0:
        flag = 2  # 2 indica la sconfitta
        
    return lives, flag

In [5]:
def check_letter(l):

    """
    Questa funzione prende in input una lettera e verifica se è una consonante o una vocale: se consonante, la restituisce tale in una lista;
    se vocale, restituisce una lista che contiene sé stessa e le possibili forme accentate presenti nelle parole italiane.
    ---------
    Parameters:
    l, str
        Lettera
    ---------
    Returns:
    l, list
        Lista che contiene la lettera se consonante oppure lista che contiene la lettera e alcune sue forme accentate se vocale
    ---------
    Richiede verifica di input ('l' deve essere un carattere alfabetico).
    """

    if l in "aeiou":
        if l == "a":
            l = ["a","à"]
        elif l == "e":
            l = ["e","è","é"]
        elif l == "i":
            l == ["i","ì"]
        elif l == "o":
            l = ["o","ò"]
        elif l == "u":
            l = ["u","ù"]
    else: l = list(l)

    return l


In [6]:
def check_single_letter(user_input, secret_word, lives, flag):
        
    # Richiamiamo la funzione check_letter per ottenere le varianti (es: se 'e' -> ['e', 'è', 'é']) (restituisce una lista)
    letter_variants = check_letter(user_input)
        
    # Verifichiamo se almeno una delle varianti (es: 'a' o 'à') è presente nella parola segreta
    letter_found = False
    for let in letter_variants:
        if let in secret_word:
            letter_found = True
            break # Basta trovarne una per considerare il tentativo corretto
            
    else:
        lives, flag = handle_error(lives, flag) # Se letter_found è rimasto false, la lettera non è presente nella parola segreta e 
                                                # richiamiamo la funzione per la gestione delle vite
        
    # Restituiamo anche 'letter_variants' (es: ['a', 'à']) così il ciclo principale 
    # sa quali esatte lettere andare a svelare nella stringa nascosta.
    return lives, flag, letter_variants, letter_found

In [7]:
def check_full_word(user_input, secret_word, lives, flag):
    
    word_found = False
    if user_input == secret_word:
        word_found = True  #variabile per tracciare solo l'esito del turno
        print("Hai indovinato la parola!")
        flag = 1  # 1 indica la vittoria
        return lives, flag, word_found
    else:
        print("Parola errata!")
        lives, flag = handle_error(lives, flag)
        return lives, flag, word_found

In [8]:
def check_input(user_input, secret_word, lives, flag):
    """
    Funzione principale gestione input: acquisisce l'input, lo pulisce, ne verifica 
    la validità formale e lo smista verso la corretta funzione di controllo.

    ---------
    Parametri:
    - user_input: la lettera singola inserita dall'utente.
    - secret_word: la parola segreta da indovinare.
    - lives: il numero di vite/tentativi rimasti.
    - flag: lo stato attuale della partita (0 = in corso, 1 = vittoria, 2 = sconfitta).

    ---------
    Logica di funzionamento:
    1. Pulizia iniziale: Rimuove spazi accidentali e converte in minuscolo (.strip().lower()).
    2. Controllo Alfabetico Globale: Usa .isalpha() per bloccare numeri, simboli o stringhe 
       miste. In caso di errore, applica una penalità (handle_error) e interrompe il turno.
    3. Smistamento per Lunghezza:
       - Singola lettera (len == 1): Applica un filtro .isascii() per bloccare l'inserimento 
         diretto di lettere accentate o speciali (es. ç, à). Se il carattere è valido, 
         delega la ricerca alla funzione check_single_letter.
       - Parola intera (len > 1): Omette volutamente il controllo ASCII per consentire il 
         tentativo con parole italiane accentate (es. "città") e delega il confronto 
         alla funzione check_full_word.

    ---------
    Restituisce:
    - lives: il numero di vite aggiornate.
    - flag: lo stato della partita dopo il turno (0, 1 o 2).
    - letter_variants: le varianti della lettera da svelare a schermo (None per le parole intere o errori).
    - word_found: l'esito del singolo turno in formato booleano (True/False).
    """
    # Puliamo l'input da eventuali spazi accidentali e rendiamolo minuscolo
    user_input = user_input.strip().lower()

    # CONTROLLO ALFABETICO GLOBALE
    if not user_input.isalpha():
        print("Input non valido: inserisci almeno una lettera e solo lettere dell'alfabeto italiano.")
        lives, flag = handle_error(lives, flag)
        
        # Restituisce Vite, Flag, varianti (Nessuna), successo (Falso)
        return lives, flag, None, False

    # SMISTAMENTO LUNGHEZZA: Lettera singola o parola intera
    if len(user_input) == 1:

        if not user_input.isascii():
            print("Input non valido: inserisci solo lettere standard e non accentate.")
            lives, flag = handle_error(lives, flag)
            # Restituiamo i 4 valori
            return lives, flag, None, False
        
        # Se passa il controllo, richiama la funzione per la singola lettera
        lives, flag, letter_variants, letter_found = check_single_letter(user_input, secret_word, lives, flag)
        return lives, flag, letter_variants, letter_found
        
    else:
        # Richiama la funzione per la parola intera
        lives, flag, word_found = check_full_word(user_input, secret_word, lives, flag)
        
        # Restituiamo 4 valori: Vite, Flag, varianti (Nessuna), vittoria (Vero se flag == 1)
        return lives, flag, None, word_found

In [9]:
def ask_play_again():
    """Se l'utente, a partita finita, vuole giocarne un'altra: ``True`` per sì,
    ``False`` per no.

    Le risposte riconosciute sono le stesse di ``ask_to_play`` (``s``, ``si``,
    ``sì``, ``n``, ``no``, senza badare a maiuscole e spazi); cambia solo la
    domanda. Le due funzioni restano separate perché stanno in due punti
    diversi del gioco, e il codice principale si legge meglio se il nome dice
    in quale dei due ci si trova.

    **Chi la usa.** Il codice principale, quando ``flag`` diventa 1 (vittoria)
    o 2 (sconfitta). Con ``True`` si ricomincia dalla scelta della difficoltà:
    vite, ``flag`` e parola nascosta vanno riportati al valore iniziale, o la
    nuova partita erediterebbe lo stato di quella appena finita.
    """
    while True:
        risposta = input("Vuoi giocare di nuovo? (s/n): ").strip().lower()

        if risposta in ["s", "si", "sì"]:
            return True
        if risposta in ["n", "no"]:
            return False

        # Anche una riga vuota (solo Invio) arriva qui: non è né un sì né un no.
        print("Input non valido: rispondi s (sì) oppure n (no).")

In [10]:
def stampa_richiesta_giocare():
    print("Vuoi giocare a L'Impiccato? (si/no): ")

def stampa_iniziale():
    print("=========================================")
    print("       BENVENUTO A L'IMPICCATO!          ")
    print("=========================================")

def stampa_scegli_difficolta():
    print("Scegli la difficoltà del gioco: ")
    print("1 - Facile")
    print("2 - Difficile")

def stampa_errore_lettera():
    print("Errore: Devi inserire una singola lettera valida dell'alfabeto.")

def stampa_errore_parola():
    print("Errore: La parola inserita non è in un formato valido.")

def stampa_errore_si_no():
    print("Errore: Input non valido. Rispondi solo con 'si' o 'no'.")

def stampa_errore_1_2():
    print("Errore: Scelta non valida. Inserisci solo '1' o '2'.")

def stampa_esito_parola(indovinata: bool):
    if indovinata:
        print("Hai indovinato l'intera parola!")
    else:
        print("Sbagliato! Non è questa la parola. Hai perso un tentativo.")

def stampa_esito_lettera(indovinata: bool):
    if indovinata:
        print("Ottima scelta! La lettera è presente nella parola.")
    else:
        print("Peccato! La lettera non c'è.")

def creazione_impiccato(numero_vite:int):
    impiccato_stato = {}
    impiccato_stato[6] = "____________\n|          |\n|           \n|           \n|           \n|           \n|           \n|           \n|_____"
    impiccato_stato[5] = "____________\n|          |\n|         _|\n|        |xx|\n|           \n|           \n|           \n|           \n|_____"
    impiccato_stato[4] = "____________\n|          |\n|         _|\n|        |xx|\n|         ||\n|           \n|           \n|           \n|_____"
    impiccato_stato[3] = "____________\n|          |\n|         _|\n|        |xx|\n|         ||\ \n|           \n|           \n|           \n|_____"
    impiccato_stato[2] = "____________\n|          |\n|         _|\n|        |xx|\n|        /||\ \n|           \n|           \n|           \n|_____"
    impiccato_stato[1] = "____________\n|          |\n|         _|\n|        |xx|\n|        /||\ \n|          \ \n|           \n|           \n|_____"
    impiccato_stato[0] = "____________\n|          |\n|         _|\n|        |xx|\n|        /||\ \n|         /\ \n|           \n|           \n|_____"
    return impiccato_stato[numero_vite]

def stampa_stato_partita(numero_vite:int, parola_nascosta: list, 
                     lettere_usate: set):
    impiccato = creazione_impiccato(numero_vite)
    print("Stato corrente della partita:")
    print(impiccato)
    parola_formattata = " ".join(parola_nascosta) 
    print("Parola: ", parola_formattata)
    lettere_formattate = ", ".join(sorted(lettere_usate))
    print("Lettere usate:", lettere_formattate)


def stampa_game_over(vittoria: bool, parola_segreta: str = ""):
    print("\n=============== GAME OVER ===============")
    if vittoria:
        print("COMPLIMENTI! Hai vinto la partita!")
    else:
        print("HAI PERSO! L'omino è stato impiccato.")
        if parola_segreta:
            print("La parola segreta era: {parola_segreta.upper()}")
    print("=========================================")

def stampa_richiesta_rigiocare():
    print("Vuoi fare un'altra partita? (si/no): ")


# ==========================================
# FUNZIONE CENTRALE
# ==========================================
def gestore_messaggi(tipo_messaggio: str, selettore_esito: bool = False, parola_nascosta: list | None = None, 
                     lettere_usate: set | None = None, numero_vite: int = 6):
    """
    Funzione centrale che smista le chiamate alle funzioni di stampa corrette in basa al tipo di maessaggio passato come stringa.
    Tipi di messaggi gestiti:
    - richiesta se vuole giocare
    - stampa iniziale grafica gioco
    - scegli la difficoltà
    - errore input:
        - Errore lettera
        - Errore parola
        - Errore domande risposta si/no
        - Errore domanda risposta 1/2
    - indovina o no la parola intera
    - indovina o no la lettera
    - stati dell'impiccato, stato parola, insieme lettere dette
    - game over (vittoria, sconfitta)
    - rigiocare la partita

    tipo_messaggio: stringa che identifica il tipo di messaggio da stampare.
    selettore_esito: valore booleano che identifica per le risposte che hanno 2 varianti quale usare (di default False se non usato)
    parola_nascota: lista contenente lo stato della parola nascosta da stampare (di default None se non usato)
    lettere_usate: set contenente le lettere già usate da stampare (di default None se non usato)
    numero_vite: vite rimanenti al giocatore, serve per stampare lo stato dell'impiccato graficamente (di default 6 se non usato)

    """
    match tipo_messaggio:
        case "richiesta_giocare":
            stampa_richiesta_giocare()
        case "stampa_iniziale":
            stampa_iniziale()
        case "scegli_difficolta":
            stampa_scegli_difficolta()
        case "errore_lettera":
            stampa_errore_lettera()
        case "errore_parola":
            stampa_errore_parola()
        case "errore_si_no":
            stampa_errore_si_no()
        case "errore_1_2":
            stampa_errore_1_2()
        case "esito_parola":
            stampa_esito_parola(selettore_esito)
        case "esito_lettera":
            stampa_esito_lettera(selettore_esito)
        case "stato_gioco":
            stampa_stato_partita(numero_vite, parola_nascosta, lettere_usate)
        case "game_over":
            stampa_game_over(selettore_esito)
        case "richiesta_rigiocare":
            stampa_richiesta_rigiocare()

In [11]:
def main():
    # Mostra il messaggio di benvenuto iniziale
    gestore_messaggi("stampa_iniziale")
    
    # Chiede all'utente se vuole iniziare a giocare (True/False)
    gioca = ask_to_play()
    
    while gioca:
        # Sceglie la difficoltà ("facile" o "difficile")
        difficolta_scelta = choose_difficulty()
        # Genera la parola casuale in base alla difficoltà scelta
        secret_word = random_word(difficolta_scelta)
        
        # Inizializzazione dello stato della singola partita
        lives = 6  # Impostato a 6 per combaciare con gli stati grafici dell'impiccato
        flag = 0   # 0 = in corso, 1 = vittoria, 2 = sconfitta
        parola_nascosta = ["_"] * len(secret_word)
        lettere_usate = set()
        
        while flag == 0:
            # Stampa lo stato corrente del gioco (grafica impiccato, parola con trattini, lettere usate)
            gestore_messaggi("stato_gioco", numero_vite=lives, parola_nascosta=parola_nascosta, lettere_usate=lettere_usate)
            
            # Richiesta input all'utente
            user_input = input("Inserisci una lettera o prova a indovinare l'intera parola: ").strip().lower()
            
            # Tracciamo l'utilizzo delle lettere per mostrarle a schermo
            if len(user_input) == 1 and user_input.isalpha():
                lettere_usate.add(user_input)
            
            # Smistamento e controllo dell'input
            lives, flag, letter_variants, is_found = check_input(user_input, secret_word, lives, flag)
            
            # Se è stata inserita una lettera singola corretta, aggiorniamo la parola nascosta
            if letter_variants is not None and is_found:
                for i, char in enumerate(secret_word):
                    if char in letter_variants:
                        parola_nascosta[i] = char
                
                # Verifica automatica di vittoria se non ci sono più trattini da svelare
                if "_" not in parola_nascosta:
                    flag = 1 
                    
        # Gestione del Game Over una volta usciti dal loop del singolo turno
        if flag == 1:
            gestore_messaggi("game_over", selettore_esito=True)
        elif flag == 2:
            gestore_messaggi("stato_gioco", numero_vite=lives, parola_nascosta=parola_nascosta, lettere_usate=lettere_usate)
            gestore_messaggi("game_over", selettore_esito=False)
            # La parola segreta non è supportata da gestore_messaggi in fase di chiamata, la stampiamo a parte
            print(f"La parola segreta era: {secret_word.upper()}") 
            
        # Richiesta di riavvio della partita
        gioca = ask_play_again()
        
    if gioca:
        print("Grazie per aver giocato, alla prossima!")
    else:
        print("Sarà per la prossima volta!")
